# 08 — The Payoff: Multi-Agent with the Official SDK

## Why this notebook exists

For seven notebooks we hand-rolled every byte of A2A. JSON-RPC envelopes, task state machines, SSE frames, webhook signatures, OAuth flows. The goal was making the protocol *legible*: if you've stuck with the series, you can now look at any A2A interaction in the wild and know what's happening.

**You don't want to keep writing that code.** In production you reach for an SDK that handles the protocol mechanics so your code can focus on the agent's behavior.

This notebook installs Google's official **`a2a-sdk`**, rebuilds the researcher and writer from notebook 01 using the SDK's `AgentExecutor` pattern, and orchestrates them with an SDK-driven coordinator. It closes with a side-by-side diff that shows how much hand-rolling the SDK absorbs.

> *Pins `a2a-sdk==0.3.26` (the v0.3 compatibility line). The 1.x line targets the A2A 1.0 spec, which has wire-format differences from what we've been building.*

## What you'll learn

- How the `a2a-sdk` decomposes an A2A server: `AgentExecutor` (your code), `DefaultRequestHandler` (protocol glue), `InMemoryTaskStore` (state), and an `A2AStarletteApplication` that wires the routes into Starlette.
- How to build a typed `AgentCard` programmatically with the SDK's classes.
- How an `AgentExecutor` enqueues `TaskStatusUpdateEvent` and `TaskArtifactUpdateEvent` instances — the same events you built by hand in notebook 05, just constructed via the SDK's types (or its `TaskUpdater` helper).
- How an A2A *client* (the coordinator) uses `A2ACardResolver` to discover an agent and then sends typed `SendMessageRequest` envelopes.
- Why the side-by-side line count matters: it's the cost of *not* having a standard.
- Where to go next: real LLMs behind the agents, multi-language interop, production deployment.

## 1. Setup

Install `a2a-sdk` (pinned to the v0.3-compatible line) and import the helpers we'll use throughout.

In [ ]:
import sys
import subprocess

# The Starlette + SSE wiring lives in the `[http-server]` extras, so install both
# the SDK and that extras bundle to get a server we can stand up in-notebook.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "a2a-sdk[http-server]==0.3.26"])
print("a2a-sdk installed")

In [ ]:
import asyncio
import threading
import time
import uuid

import httpx
import uvicorn

# SDK server surfaces. In a2a-sdk 0.3.26 the symbols live where you'd expect,
# though some names differ from earlier plan drafts (notably no separate
# `create_*_routes` helpers — the Starlette app builder owns the routes).
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.server.apps import A2AStarletteApplication

# Typed wire models — same shape as the JSON we hand-rolled, just as pydantic
# classes. Note `AgentInterface(transport=..., url=...)` and that `TaskState`
# is a plain string enum (TaskState.working, .completed, .failed), not a
# protobuf-wrapped value.
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentSkill,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    Task,
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
    TextPart,
)

# SDK helpers for building messages, artifacts, and fresh tasks from a user
# message — the building blocks our hand-rolled code spelled out by hand.
from a2a.utils import (
    new_agent_text_message,
    new_task,
    new_text_artifact,
)

# Client surfaces.
from a2a.client import A2ACardResolver, A2AClient

# Background-thread uvicorn helper (same as previous notebooks).
_servers: list[uvicorn.Server] = []


def run_server_in_thread(app, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")
    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

## 2. A Tour of the SDK

| SDK piece | What it replaces (vs. our hand-rolled code) |
|---|---|
| `AgentExecutor` (subclass it; implement `async execute(ctx, queue)`) | The whole `handle_jsonrpc` function from notebook 03 — method dispatch, task creation, status updates, artifact assembly. |
| `RequestContext` | Parsed `Message`, `task_id`, `context_id` — instead of plucking them out of `req.params["message"]` by hand. `context.get_user_input()` even joins all text parts for you. |
| `EventQueue` | Replaces the `_set_status` / `_set_artifacts` lock-protected stores plus the SSE generator — you `await event_queue.enqueue_event(thing)` and the SDK handles delivery for both `message/send` (last event wins) and `message/stream` (streamed). |
| `TaskUpdater` (helper around `EventQueue`) | The bookkeeping for "transition this task to working / add this artifact / mark complete" — fewer fields to set by hand. |
| `DefaultRequestHandler` | The router that maps JSON-RPC method names to `AgentExecutor` callbacks plus task lifecycle plumbing (the entire notebook 04 state machine). |
| `InMemoryTaskStore` | Our `TASKS` dict + `STORE_LOCK` from notebook 04. |
| `A2AStarletteApplication(...).build()` | The FastAPI route declarations from notebooks 02 and 03 — mounts the agent-card endpoint at `/.well-known/agent-card.json` and the JSON-RPC endpoint at `/`. |
| `AgentCard`, `AgentCapabilities`, `AgentSkill`, `AgentInterface` (types) | Our hand-written pydantic `AgentCard` model from notebook 02. |
| `new_task`, `new_text_artifact`, `new_agent_text_message` (helpers) | Boilerplate task/artifact/message construction. |
| `TaskStatusUpdateEvent`, `TaskArtifactUpdateEvent` (types) | Our hand-written event models from notebook 05. |
| `A2ACardResolver`, `A2AClient` (client side) | The hand-rolled JSON-RPC envelope construction + `httpx.post` calls. |

Same protocol on the wire — what the SDK does is absorb the *mechanics* of producing and consuming it. Field names in Python switch from `defaultInputModes` (the JSON wire shape we've been using) to `default_input_modes` (the SDK's snake_case Python convention); the SDK serializes back to the spec-correct camelCase on the wire.

## 3. The Researcher, SDK-Built

The `ResearcherAgent` is just a callable that looks up facts. The `ResearcherAgentExecutor` is the SDK adapter — given a `RequestContext` and an `EventQueue`, it enqueues the new `Task`, marks it `working`, runs the agent, then enqueues the artifact and a `completed` status update. Compare cell-by-cell against `_handle_message_send` from notebook 03 to see what the SDK absorbs.

In [ ]:
FACTS_BY_TOPIC: dict[str, list[str]] = {
    "octopuses": [
        "Octopuses have three hearts.",
        "They can change color in under a second.",
        "Each of their arms has its own neural cluster.",
    ],
    "rome": [
        "Rome was founded in 753 BCE according to tradition.",
        "The Roman Empire at its peak spanned roughly 5 million km².",
        "Roman concrete used volcanic ash and is still studied today.",
    ],
}


class ResearcherAgent:
    """Pure-Python agent logic - knows nothing about A2A."""

    async def invoke(self, topic: str) -> list[str] | None:
        return FACTS_BY_TOPIC.get(topic.strip().lower())


class ResearcherAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = ResearcherAgent()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # First request for this message: synthesize a Task and tell the
        # framework about it. (Subsequent turns would arrive with
        # context.current_task already populated.)
        task = context.current_task
        if task is None:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)

        # TaskUpdater wraps the EventQueue with the right task_id / context_id
        # already plumbed in, so we don't have to repeat them on every event.
        updater = TaskUpdater(event_queue, task.id, task.context_id)

        await updater.update_status(
            TaskState.working,
            message=new_agent_text_message("Looking up facts…"),
        )

        topic = context.get_user_input()
        facts = await self.agent.invoke(topic)

        if facts is None:
            await updater.failed(
                message=new_agent_text_message(f"No facts on file for {topic!r}."),
            )
            return

        artifact = new_text_artifact(
            name=f"facts-about-{topic.lower()}",
            text="\n".join(facts),
        )
        await updater.add_artifact(artifact.parts, name=artifact.name)
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("cancel not supported in this demo")


researcher_card = AgentCard(
    name="Researcher",
    description="Returns canned facts on a small set of well-known topics.",
    version="0.1.0",
    url="http://127.0.0.1:8010/",
    preferred_transport="JSONRPC",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[AgentSkill(
        id="research_topic",
        name="Research a topic",
        description="Given a topic, return facts about it.",
        tags=["research"],
        examples=["octopuses", "rome"],
    )],
)

researcher_handler = DefaultRequestHandler(
    agent_executor=ResearcherAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

researcher_app = A2AStarletteApplication(
    agent_card=researcher_card,
    http_handler=researcher_handler,
).build()

researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Researcher (SDK) on http://127.0.0.1:8010")

## 4. The Writer, SDK-Built

The writer takes a topic and a list of facts (joined by newlines) and emits a one-paragraph summary. Pure stub — no LLM. Built with the same SDK pattern as the researcher.

In [ ]:
class WriterAgent:
    async def invoke(self, topic: str, fact_lines: str) -> str:
        facts = [line.strip() for line in fact_lines.splitlines() if line.strip()]
        return f"Here is what we know about {topic}: " + " ".join(facts)


class WriterAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = WriterAgent()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task
        if task is None:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)

        # Input convention for this demo: message text is "<topic>\n---\n<fact_lines>".
        raw = context.get_user_input()
        if "\n---\n" not in raw:
            await updater.failed(
                message=new_agent_text_message("Writer expects '<topic>\\n---\\n<facts>'."),
            )
            return
        topic, fact_lines = raw.split("\n---\n", 1)

        paragraph = await self.agent.invoke(topic.strip(), fact_lines)

        artifact = new_text_artifact(name="summary", text=paragraph)
        await updater.add_artifact(artifact.parts, name=artifact.name)
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("cancel not supported in this demo")


writer_card = AgentCard(
    name="Writer",
    description="Turns a topic + facts into a one-paragraph summary.",
    version="0.1.0",
    url="http://127.0.0.1:8011/",
    preferred_transport="JSONRPC",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[AgentSkill(
        id="write_summary",
        name="Write a summary",
        description="Given a topic and facts, return a one-paragraph summary.",
        tags=["writing"],
        examples=["octopuses\\n---\\nThey have three hearts."],
    )],
)

writer_handler = DefaultRequestHandler(
    agent_executor=WriterAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

writer_app = A2AStarletteApplication(
    agent_card=writer_card,
    http_handler=writer_handler,
).build()

writer_server = run_server_in_thread(writer_app, port=8011)
print("Writer (SDK) on http://127.0.0.1:8011")

## 5. The Coordinator

The coordinator is an A2A *client* — it doesn't expose its own endpoint. It uses the SDK's `A2ACardResolver` to fetch each agent's card (proving discovery works), then sends a `message/send` request to each, picks the result artifact out of the response, and chains them: researcher's text becomes part of the writer's input.

The SDK still wants a typed `SendMessageRequest` wrapping a `Message` wrapping `Part`s — these are exactly the wire shapes we hand-built in notebook 03, just as pydantic models.

In [ ]:
def _build_user_message(text: str) -> Message:
    """Wrap user text in the SDK's typed Message envelope."""
    return Message(
        role=Role.user,
        message_id=str(uuid.uuid4()),
        parts=[Part(root=TextPart(text=text))],
    )


async def call_agent(base_url: str, user_text: str) -> str:
    """Send `user_text` to the agent at `base_url`; return its first artifact's text."""
    async with httpx.AsyncClient(timeout=30.0) as http:
        # Discovery: hit /.well-known/agent-card.json and parse it into AgentCard.
        resolver = A2ACardResolver(httpx_client=http, base_url=base_url)
        card = await resolver.get_agent_card()

        # The SDK ships several clients; A2AClient is the simple JSON-RPC one
        # that maps closest to what we hand-rolled in notebook 03. (It prints a
        # DeprecationWarning suggesting ClientFactory — same wire format either
        # way; we stick with the simpler surface for the demo.)
        client = A2AClient(httpx_client=http, agent_card=card)

        request = SendMessageRequest(
            id=str(uuid.uuid4()),
            params=MessageSendParams(message=_build_user_message(user_text)),
        )
        response = await client.send_message(request)

    # SendMessageResponse is a RootModel wrapping either a success or an error.
    # On success, .root.result is the Task with our artifacts attached.
    result = response.root.result
    if not isinstance(result, Task):
        raise RuntimeError(f"Expected Task from {base_url}, got {type(result).__name__}")
    if not result.artifacts:
        raise RuntimeError(f"No artifacts in response from {base_url}")
    # Each Part is a RootModel too; .root.text exposes the underlying TextPart.
    return result.artifacts[0].parts[0].root.text


async def coordinate(topic: str) -> str:
    facts_text = await call_agent("http://127.0.0.1:8010", topic)
    writer_input = f"{topic}\n---\n{facts_text}"
    paragraph = await call_agent("http://127.0.0.1:8011", writer_input)
    return paragraph


print("Coordinator defined.")